# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is available through a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets (`@id`), their fields, and field `@id`s.

In [ ]:
# List all record sets in the dataset by their @id
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
else:
    record_sets = dataset.list_record_sets()

print("Record Sets (@id):")
record_set_ids = []
for rs in record_sets:
    print(f"  {rs['@id']}  |  Name: {rs['name'] if 'name' in rs else '<no name>'}")
    record_set_ids.append(rs['@id'])
    print("    Fields:")
    for f in rs['fields']:
        fid = f['@id'] if isinstance(f, dict) else f
        print(f"      {fid}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use record set and field `@id`s retrieved above.

In [ ]:
# Load each record set as a DataFrame, storing by Record Set @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"  (No records found for {record_set_id})\n")

# For demonstration, pick the first record set with data
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

assert main_record_set_id is not None, "No record set has data."

print(f"\nMain record set selected: {main_record_set_id}")
print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates cleaning, grouping, and transformation with field references by `@id`.

In [ ]:
# Display all columns for reference
print('Columns in selected record set:')
print(dataframes[main_record_set_id].columns.tolist())

# --- Example of field selection by @id ---
# Try to select a numeric field by its @id, e.g., for age at diagnosis or a similar field
import numpy as np

df = dataframes[main_record_set_id]

# Heuristic: select first numeric column (int/float)
numeric_col = None
for c in df.columns:
    if np.issubdtype(df[c].dropna().apply(type).mode()[0], (int, float, np.integer, np.floating)):
        numeric_col = c
        break
if numeric_col is None:
    # Fallback: pick a column containing 'Age' or similar
    for c in df.columns:
        if "age" in c.lower():
            numeric_col = c
            break

if numeric_col is None:
    raise ValueError("No numeric field found. Please update 'numeric_col' for analysis.")
print(f"Using numeric field for EDA: {numeric_col}")

# Clean numeric field for numeric analysis
df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')

threshold = df[numeric_col].mean() if not df[numeric_col].isnull().all() else 0
print(f"Threshold for {numeric_col}: {threshold:.2f}")
filtered_df = df[df[numeric_col] > threshold]
print(f"Filtered records with {numeric_col} > {threshold:.2f}:")
print(filtered_df[[numeric_col]].head())

# Normalize numeric values in filtered data
if filtered_df[numeric_col].std() > 0:
    filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
    print(f"Normalized {numeric_col} for filtered records:")
    print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())
else:
    print(f"Standard deviation of {numeric_col} is zero; skipping normalization.")

# Attempt to group by a likely categorical field, e.g., sex, anatomical site, or similar
group_candidates = [c for c in df.columns if (df[c].dtype == 'object' and len(df[c].unique()) < min(10, len(df)//10))]
group_field = group_candidates[0] if group_candidates else None

if group_field:
    print(f"\nGrouping on: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_col].mean()
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_col].dropna(), kde=True, bins=15)
plt.title(f'Histogram of {numeric_col} in {main_record_set_id}')
plt.xlabel(numeric_col)
plt.show()

# If grouped_df is defined, show a boxplot
if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_col, data=df)
    plt.title(f'{numeric_col} by {group_field}')
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library, referencing all relevant record sets and fields by their `@id`. We showed how to filter and normalize numeric fields and visualize key relationships. For further analysis, consult the Croissant schema for detailed descriptions of each field's meaning and data provenance.